# 🌙 Deploy Moon AI Backend on Free Cloud GPU (Google Colab)
### High-Speed NVIDIA T4 GPU Inference Engine for Moon AI
This notebook runs the **Moon AI FastAPI Backend** on a **free NVIDIA GPU (T4/A100)** in Google Colab.
It uses **ngrok** to provide a secure public HTTPS URL that connects directly to your **Vercel Frontend**.

---
### ⚡ Speed Comparison:
* **Your Laptop CPU**: 0.1 tokens/sec (3,103s / ~51 mins)
* **Colab NVIDIA T4 GPU**: **50–80 tokens/sec (1.5 to 3 seconds!)** — **~500x faster!**

In [ ]:
# Step 1: Verify NVIDIA GPU is active
!nvidia-smi

In [ ]:
# Step 2: Install Fast Cloud Dependencies (Prebuilt CUDA wheels for instant setup)
!pip install -q fastapi uvicorn pyngrok pydantic python-multipart tiktoken huggingface_hub
# Prebuilt CUDA 12.2 wheel for llama-cpp-python (installs in 5 seconds instead of 15 min compile!)
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 --upgrade --force-reinstall --no-cache-dir

In [ ]:
# Step 3: Clone / Update Moon AI Repository
import os
if not os.path.exists('/content/moon-ai'):
    !git clone https://github.com/Prem759-0/NanoGPT.git /content/moon-ai
%cd /content/moon-ai
!git pull || echo "Repository directory active"

In [ ]:
# Step 4: Download 7.6B GGUF Model (High-Speed HuggingFace CDN)
import os
model_file = "Qwen2.5-Coder-7B-Instruct.Q4_K_M.gguf"
if not os.path.exists(model_file):
    print("📥 Downloading 7.6B GGUF Model via HuggingFace CLI...")
    !huggingface-cli download Qwen/Qwen2.5-Coder-7B-Instruct-GGUF qwen2.5-coder-7b-instruct-q4_k_m.gguf --local-dir . --local-dir-use-symlinks False
    if os.path.exists("qwen2.5-coder-7b-instruct-q4_k_m.gguf"):
        os.rename("qwen2.5-coder-7b-instruct-q4_k_m.gguf", model_file)

print(f"✅ Model file verified: {model_file} ({os.path.getsize(model_file)/1e9:.2f} GB)")

In [ ]:
# Step 5: Launch ngrok Public Tunnel & Moon AI FastAPI Server
import os
from pyngrok import ngrok

# Your ngrok Auth Token
NGROK_AUTH_TOKEN = "37jwOSfLYd9gfleSnVI8BsHz8xC_Y3Kng7dshpfjX97Dvv8v"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Reset any existing tunnels
try:
    ngrok.kill()
except Exception:
    pass

# Open secure HTTPS tunnel to port 8000
public_tunnel = ngrok.connect(8000)
public_url = public_tunnel.public_url

print("=" * 75)
print("🚀 MOON AI CLOUD GPU BACKEND IS NOW ONLINE!")
print("=" * 75)
print(f"\n👉 YOUR PUBLIC CLOUD API URL:\n   {public_url}\n")
print("👉 Copy the URL above and paste it into Vercel Settings:")
print(f"   Environment Variable: VITE_API_URL = {public_url}\n")
print("=" * 75)

# Set GPU execution flag and launch server
os.environ["USE_GPU"] = "1"
!python -m uvicorn webapp.main:app --host 0.0.0.0 --port 8000